In [ ]:
import Pkg
Pkg.add(["JuMP", "GLPK", "DataFrames", "CSV", "Statistics"])
import Pkg; Pkg.add("HiGHS")

In [1]:
# ============================================================
#   Optimal Classification Tree (OCT) — Breast Cancer Dataset
#   Based on: Bertsimas & Dunn (2017) "Optimal Classification Trees"
#   Project 6 — Mixed Integer Linear Programming formulation
#
#   WHAT THIS CODE DOES:
#   Instead of building a decision tree greedily (like CART, which
#   makes locally optimal splits one at a time), OCT formulates the
#   entire tree-building problem as a single MILP and finds the
#   GLOBALLY optimal tree — the one that minimises total training
#   misclassification across all nodes simultaneously.
#
#   INSTANCE PARAMETERS:
#   • n = 175 samples  (stratified subsample of 569)
#   • D = 2            (tree depth: 3 branch nodes, 4 leaves)
#   • p = 5 features   (top 5 by Fisher discriminant score)
#   • α = 0.0001       (complexity penalty per split)
#   • MIP gap = 0.5%   (solver stops within 0.5% of proven optimal)
#   These parameters are tuned to give a ~2 minute solve time,
#   which qualifies as a "medium-sized" instance.
# ============================================================

# ── Package imports ──────────────────────────────────────────
# JuMP     : algebraic modelling language for optimisation in Julia
# HiGHS    : open-source MILP solver used to solve the model
# CSV/DataFrames : data loading
# Printf   : formatted output (%d, %.2f etc.)
# Random   : reproducible random subsampling
using JuMP, HiGHS, LinearAlgebra, Statistics, CSV, DataFrames, Printf, Random

# ─────────────────────────────────────────────────────────────
# SECTION 1: Load data & prepare the training set
# ─────────────────────────────────────────────────────────────

# Load the breast cancer CSV. Row 1 is metadata so we skip it.
# Columns 1–30 are the tumour measurements (features).
# Column 31 is the diagnosis label: 0 = malignant, 1 = benign.
file_path = joinpath(@__DIR__, "breast_cancer.csv")
data      = CSV.read(file_path, DataFrame, skipto=2, header=false)

X_full = Matrix{Float64}(data[:, 1:30])   # 569 × 30 feature matrix
y_full = Int.(Vector(data[:, 31]))         # 569-element label vector
N_full = length(y_full)                    # = 569

# ── Wall-clock timer starts here (includes all phases) ───────
t_total_start = time()

# ── Stratified subsampling ───────────────────────────────────
# We cannot use all 569 samples — the MILP would take too long.
# Stratified sampling keeps the same malignant/benign ratio as
# the full dataset (~37% malignant, ~63% benign) so the tree
# is still representative. Random.seed!(42) ensures the same
# subsample is chosen every time for reproducibility.
Random.seed!(42)
n_sub   = 150                                              # total subsample size
idx0    = shuffle(findall(y_full .== 0))                   # shuffle malignant indices
idx1    = shuffle(findall(y_full .== 1))                   # shuffle benign indices
k0      = round(Int, n_sub * sum(y_full .== 0) / N_full)  # how many malignant to keep (~65)
k1      = n_sub - k0                                       # how many benign to keep (~110)
sub_idx = vcat(idx0[1:k0], idx1[1:k1])                    # combine both class indices
X_all   = X_full[sub_idx, :]                               # subsample feature matrix
y_raw   = y_full[sub_idx]                                  # subsample labels
n       = length(y_raw)                                    # = n_sub = 150

# ── Feature selection via Fisher discriminant score ──────────
# With p=30 features the MILP is too large. We rank features by
# how well they separate the two classes using Fisher's score:
#   score_j = |mean(class1) - mean(class0)| / std(feature_j)
# A high score means the feature has very different values for
# malignant vs benign tumours relative to its spread — making it
# useful for splitting. We keep only the top 5.
mu0    = mean(X_all[y_raw .== 0, :], dims=1)[:]   # per-feature mean for malignant
mu1    = mean(X_all[y_raw .== 1, :], dims=1)[:]   # per-feature mean for benign
sigma  = std(X_all, dims=1)[:]                     # per-feature standard deviation
scores = abs.(mu1 .- mu0) ./ (sigma .+ 1e-8)      # Fisher score (1e-8 avoids /0)
feat_idx = sortperm(scores, rev=true)[1:5]          # indices of top 5 features
X_raw    = X_all[:, feat_idx]                       # keep only those 5 columns

# ── Normalise features to [0,1] ──────────────────────────────
# The OCT formulation (PDF p.13) requires xi ∈ [0,1]^p.
# We apply min-max scaling: x_scaled = (x - min) / (max - min).
# This also ensures the threshold b[t] stays in [0,1].
X_min = minimum(X_raw, dims=1)
X_max = maximum(X_raw, dims=1)
rng_f = X_max .- X_min
rng_f[rng_f .== 0] .= 1.0       # avoid division by zero for constant features
X     = (X_raw .- X_min) ./ rng_f   # final normalised feature matrix (n × p)

n, p    = size(X)
classes = [0, 1];  K = 2   # K=2 classes: malignant and benign

# Y[i,k] is the ±1 label indicator from PDF p.14:
#   Y[i,k] = +1 if sample i belongs to class k, -1 otherwise
# Used in the error counting constraints (Section D below).
Y = [y_raw[i] == classes[k] ? 1 : -1 for i in 1:n, k in 1:K]

# ── Perturbation vector ε (PDF p.14) ─────────────────────────
# MILP solvers cannot handle strict inequalities (e.g. x < b).
# The OCT formulation converts "go left if x < b" into
# "go left if x + ε ≤ b" where ε is the smallest gap between
# any two consecutive distinct values of that feature.
# This ensures no sample sits exactly on a boundary.
eps_j = zeros(p)
for j in 1:p
    sv = sort(unique(X[:, j]))                              # sorted unique values
    eps_j[j] = length(sv) > 1 ? minimum(diff(sv)) : 1e-4  # smallest gap between values
end
eps_max = maximum(eps_j)   # used in big-M for left routing constraints

# ─────────────────────────────────────────────────────────────
# SECTION 2: Define the binary tree structure
# ─────────────────────────────────────────────────────────────
# Nodes are numbered 1 to T_total using the standard binary tree
# index convention: node t has children 2t (left) and 2t+1 (right).
#
# For depth D=2:
#   T_total = 2^(D+1) - 1 = 7 nodes
#   Branch nodes T_B = {1, 2, 3}   — internal nodes that split
#   Leaf nodes   T_L = {4, 5, 6, 7} — terminal nodes that predict
#
#   Tree layout:
#              [1]
#             /   \
#           [2]   [3]
#           / \   / \
#          [4][5][6][7]
D       = 2
T_total = 2^(D+1) - 1
T_B     = 1:(T_total ÷ 2)              # branch nodes: {1, 2, 3}
T_L     = (T_total ÷ 2 + 1):T_total   # leaf nodes:   {4, 5, 6, 7}

# get_ancestors(t): for any leaf node t, returns two lists:
#   AL = ancestors reached by going LEFT  (even child index)
#   AR = ancestors reached by going RIGHT (odd  child index)
# Example: leaf 7 → path is 7→3→1
#   7 is odd  (right child of 3) → AR = [3]
#   3 is odd  (right child of 1) → AR = [3, 1]
#   AL = []
# Used to build routing constraints: every sample assigned to leaf t
# must satisfy ALL the splits along its path from root to t.
function get_ancestors(t)
    AL, AR = Int[], Int[]
    curr = t
    while curr > 1
        par = div(curr, 2)
        iseven(curr) ? push!(AL, par) : push!(AR, par)
        curr = par
    end
    return AL, AR
end

# ── Model build timer starts here ────────────────────────────
t_build_start = time()

# ─────────────────────────────────────────────────────────────
# SECTION 3: Build the MILP model
# ─────────────────────────────────────────────────────────────
# alpha: complexity penalty per split in the objective.
#   Very small (0.0001) so accuracy dominates — splits are only
#   penalised if they provide no classification benefit at all.
# M_big: "big-M" constant used in logical implication constraints.
#   Standard value is n (total number of samples). It acts as a
#   large number that "deactivates" a constraint when a binary
#   variable is 0.
alpha = 0.0001
M_big = n

# Create the JuMP model using HiGHS as the MILP solver.
# set_silent + output_flag/log_to_console: suppress all HiGHS
# console output so it doesn't flood the Jupyter cell.
model = Model(HiGHS.Optimizer)
set_silent(model)
set_optimizer_attribute(model, "output_flag",    false)
set_optimizer_attribute(model, "log_to_console", false)
# mip_rel_gap: solver stops when the gap between current best
# integer solution and the best possible lower bound is ≤ 0.5%.
# Tighter gap = longer solve but closer to proven optimal.
set_optimizer_attribute(model, "mip_rel_gap",    0.01)
# time_limit: hard safety net — returns best solution found so
# far if the gap is not closed within 300 seconds.
set_optimizer_attribute(model, "time_limit",     300.0)

# ── Decision variables (PDF p.13–14) ─────────────────────────
# a[j,t] ∈ {0,1}: equals 1 if feature j is used for the split
#                 at branch node t. At most one j can equal 1
#                 per node (enforced by constraint A below).
@variable(model, a[1:p, T_B], Bin)

# b[t] ∈ [0,1]: the split threshold at branch node t.
#               Sample goes left if feature value ≤ b[t],
#               right if feature value > b[t].
@variable(model, 0 <= b[T_B] <= 1)

# d[t] ∈ {0,1}: equals 1 if branch node t actually performs a
#               split (i.e., is active). d[t]=0 means the node
#               is a pass-through and its subtree is unused.
@variable(model, d[T_B], Bin)

# z[i,t] ∈ {0,1}: equals 1 if training sample i is assigned
#                 to leaf node t. Each sample goes to exactly
#                 one leaf (enforced by constraint B below).
@variable(model, z[1:n, T_L], Bin)

# l[t] ∈ {0,1}: equals 1 if leaf t is "active" — it receives
#               at least N_min samples. Inactive leaves make
#               no prediction.
@variable(model, l[T_L], Bin)

# c[k,t] ∈ {0,1}: equals 1 if leaf t predicts class k.
#                 Each active leaf predicts exactly one class.
@variable(model, c[1:K, T_L], Bin)

# L_err[t] ≥ 0: number of misclassified samples at leaf t.
#               The objective minimises the sum of these.
@variable(model, L_err[T_L] >= 0)

# ── Objective function (PDF p.15) ────────────────────────────
# Minimise: (1/n) × total misclassifications  +  α × number of splits
# The first term is the normalised training error.
# The second term penalises tree complexity — more splits = more
# complexity. With α=0.0001 the penalty is negligible compared to
# the error term, so the solver will always prefer accuracy.
@objective(model, Min,
    (1/n) * sum(L_err[t] for t in T_L) + alpha * sum(d[t] for t in T_B))

# ── CONSTRAINT GROUP A: Splitting structure (PDF p.13) ───────
# These constraints define valid tree structure:
#   (i)  Exactly one feature is selected at each splitting node
#        (or zero features if the node doesn't split: d[t]=0).
#   (ii) Threshold b[t] must be zero if the node doesn't split.
#   (iii)A node can only split if its parent also splits
#        (no "gaps" in the tree — splits must be contiguous from root).
for t in T_B
    @constraint(model, sum(a[j,t] for j in 1:p) == d[t])  # (i)  Σ a[j,t] = d[t]
    @constraint(model, b[t] <= d[t])                        # (ii) b[t] = 0 if d[t] = 0
    t > 1 && @constraint(model, d[t] <= d[div(t,2)])        # (iii) d[t] ≤ d[parent(t)]
end

# Force the root node (node 1) to always split.
# Without this, d[t]=0 everywhere makes all routing constraints
# trivially satisfied, allowing the solver to assign all samples
# to one leaf with zero error — a degenerate "no tree" solution.
@constraint(model, d[1] == 1)

# ── CONSTRAINT GROUP B: Sample assignment (PDF p.14) ─────────
# Each sample must be assigned to exactly one leaf.
@constraint(model, [i=1:n], sum(z[i,t] for t in T_L) == 1)

for t in T_L
    # A sample can only go to leaf t if leaf t is active.
    @constraint(model, [i=1:n], z[i,t] <= l[t])

    # An active leaf must receive at least 5 samples (N_min=5).
    # This prevents overfitting via trivial single-sample leaves.
    @constraint(model, sum(z[i,t] for i in 1:n) >= 5 * l[t])

    # Each active leaf must predict exactly one class.
    # (An inactive leaf predicts nothing: Σ c[k,t] = 0.)
    @constraint(model, sum(c[k,t] for k in 1:K) == l[t])

    # Hierarchical integrity constraint (key fix):
    # z[i,t] ≤ d[m] for all m in AR(t) — right-branch ancestors.
    # If a right-branch ancestor m does NOT split (d[m]=0), then
    # no sample can be routed rightward through m to reach leaf t.
    # Without this, d[m]=0 makes the right routing constraint
    # "0 ≥ 0 - (1-z[i,t])" trivially true, allowing free assignment.
    _, AR = get_ancestors(t)
    for m in AR
        @constraint(model, [i=1:n], z[i,t] <= d[m])
    end
end

# ── CONSTRAINT GROUP C: Routing constraints (PDF p.14) ───────
# These are the core constraints that enforce the tree logic:
# if sample i is assigned to leaf t, it must have satisfied
# every split decision along the path from the root to t.
#
# For a LEFT ancestor m (sample went left at node m):
#   a[m]'(x[i] + ε) ≤ b[m] + (1 + ε_max)(1 - z[i,t])
#   When z[i,t]=1: a[m]'(x[i] + ε) ≤ b[m]  → sample satisfies left split
#   When z[i,t]=0: constraint is relaxed (big-M deactivates it)
#
# For a RIGHT ancestor m (sample went right at node m):
#   a[m]'x[i] ≥ b[m] - (1 - z[i,t])
#   When z[i,t]=1: a[m]'x[i] ≥ b[m]  → sample satisfies right split
#   When z[i,t]=0: constraint is relaxed
for t in T_L
    AL, AR = get_ancestors(t)
    for m in AL, i in 1:n   # LEFT ancestors: feature value must be ≤ threshold
        @constraint(model,
            sum(a[j,m]*(X[i,j]+eps_j[j]) for j in 1:p) <=
            b[m] + (1+eps_max)*(1-z[i,t]))
    end
    for m in AR, i in 1:n   # RIGHT ancestors: feature value must be > threshold
        @constraint(model,
            sum(a[j,m]*X[i,j] for j in 1:p) >= b[m] - (1-z[i,t]))
    end
end

# ── CONSTRAINT GROUP D: Error counting (PDF p.15) ────────────
# For each leaf t, L_err[t] counts the number of misclassified
# samples — samples that are assigned to leaf t but do NOT belong
# to the majority class predicted by that leaf.
#
# N_t   = total samples at leaf t  = Σ_i z[i,t]
# N_kt  = samples of class k at t  = ½ Σ_i (1 + Y[i,k]) z[i,t]
#         (Y[i,k]=+1 when y_i=k, so (1+Y[i,k])/2 = 1 if same class, 0 otherwise)
# L_err[t] = N_t - N_kt  when leaf t predicts class k (c[k,t]=1)
#
# Both bounds are needed:
#   LOWER: L_err[t] ≥ N_t - N_kt - M(1-c[k,t])
#          → forces L_err up to the correct miscount when c[k,t]=1
#   UPPER: L_err[t] ≤ N_t - N_kt + M(1-c[k,t])
#          → pins L_err exactly to the miscount when c[k,t]=1
#          → without this, solver could set L_err=0 artificially
for t in T_L
    N_t = @expression(model, sum(z[i,t] for i in 1:n))
    for k in 1:K
        N_kt = @expression(model, sum(0.5*(1+Y[i,k])*z[i,t] for i in 1:n))
        @constraint(model, L_err[t] >= N_t - N_kt - M_big*(1-c[k,t]))  # lower bound
        @constraint(model, L_err[t] <= N_t - N_kt + M_big*(1-c[k,t]))  # upper bound
    end
end

# ─────────────────────────────────────────────────────────────
# SECTION 4: Solve the MILP
# ─────────────────────────────────────────────────────────────
# Record model build time (constraint generation overhead in JuMP)
t_build = round(time() - t_build_start, digits=2)

# optimize!() calls HiGHS to solve the MILP via branch-and-bound:
# 1. Solves LP relaxation (binary variables treated as continuous)
# 2. Branches on fractional variables, creating sub-problems
# 3. Prunes branches provably worse than current best solution
# 4. Stops when MIP gap ≤ 0.5% or time_limit is reached
t0 = time()
optimize!(model)
elapsed = round(time() - t0, digits=2)       # pure solver time
t_total = round(time() - t_total_start, digits=2)  # total wall-clock time

# ─────────────────────────────────────────────────────────────
# SECTION 5: Extract results and print output
# ─────────────────────────────────────────────────────────────
# All 30 original feature names in column order
all_feature_names = [
    "radius_mean","texture_mean","perimeter_mean","area_mean","smoothness_mean",
    "compactness_mean","concavity_mean","concave_pts_mean","symmetry_mean","fractal_dim_mean",
    "radius_se","texture_se","perimeter_se","area_se","smoothness_se",
    "compactness_se","concavity_se","concave_pts_se","symmetry_se","fractal_dim_se",
    "radius_worst","texture_worst","perimeter_worst","area_worst","smoothness_worst",
    "compactness_worst","concavity_worst","concave_pts_worst","symmetry_worst","fractal_dim_worst"
]
sel_feature_names = all_feature_names[feat_idx]  # names of the 5 selected features

# termination_status: OPTIMAL means proven optimal within gap tolerance.
#   TIME_LIMIT means best solution found but gap not fully closed.
status  = termination_status(model)

# primal_status: checks whether a feasible integer solution was actually found.
#   FEASIBLE_POINT = yes, NEARLY_FEASIBLE_POINT = yes (minor tolerance).
#   If neither, the solver found nothing usable and we exit early.
has_sol = primal_status(model) in [MOI.FEASIBLE_POINT, MOI.NEARLY_FEASIBLE_POINT]

# Build output as a list of strings, then print with a single println().
# Single println() prevents Jupyter from truncating the output mid-way,
# which happens when many separate print calls are made.
lines = String[]

# ── Results first — most important, survives any truncation ──
push!(lines, "="^60)
push!(lines, "  OCT — Breast Cancer  |  D=$D  p=$p  n=$n")
push!(lines, "="^60)
push!(lines, @sprintf("  Solver Status : %s", status))
push!(lines, @sprintf("  Build Time    : %.2f s  (constraint generation)", t_build))
push!(lines, @sprintf("  Solve Time    : %.2f s  (%.2f min)", elapsed, elapsed/60))
push!(lines, @sprintf("  Total Time    : %.2f s  (%.2f min)  <- matches Jupyter runtime", t_total, t_total/60))

# Exit cleanly if no solution was found
if !has_sol
    push!(lines, "  ERROR: No feasible solution found.")
    println(join(lines, "\n")); exit(0)
end

push!(lines, @sprintf("  Objective Val : %.6f", objective_value(model)))
push!(lines, @sprintf("  MIP Gap       : %.3f %%", 100 * relative_gap(model)))

# ── Tree structure: read back the optimal a, b, d, l, c values ─
push!(lines, "")
push!(lines, "─"^60)
push!(lines, "  TREE STRUCTURE")
push!(lines, "─"^60)
for t in T_B
    # d[t] > 0.5 means the binary variable rounded to 1 → node splits
    if value(d[t]) > 0.5
        # Find which feature was selected (a[j,t] = 1)
        fj = findfirst(j -> value(a[j,t]) > 0.5, 1:p)
        if fj !== nothing
            # Un-normalise threshold back to original feature scale
            thr_orig = value(b[t]) * (X_max[fj] - X_min[fj]) + X_min[fj]
            push!(lines, @sprintf("  Node %d │ %-24s ≤ %.4f", t, sel_feature_names[fj], thr_orig))
        end
    else
        push!(lines, @sprintf("  Node %d │ no split", t))
    end
end
for t in T_L
    # l[t] > 0.5 means this leaf is active (received ≥ 5 samples)
    if value(l[t]) > 0.5
        # Find which class the leaf predicts
        k_idx = findfirst(k -> value(c[k,t]) > 0.5, 1:K)
        label = (k_idx !== nothing && classes[k_idx] == 0) ? "Malignant" : "Benign"
        # Count samples assigned to this leaf
        cnt   = round(Int, sum(value(z[i,t]) for i in 1:n))
        push!(lines, @sprintf("  Leaf %d │ → %-9s  (n=%d)", t, label, cnt))
    else
        push!(lines, @sprintf("  Leaf %d │ inactive", t))
    end
end

# ── Performance metrics ───────────────────────────────────────
# Reconstruct predicted label for each sample from z and c values.
# For each sample i, find the unique leaf t where z[i,t]=1,
# then read off the class prediction c[k,t]=1 for that leaf.
# break stops after finding the first active leaf to avoid
# numerical noise causing multiple z[i,t] ≈ 1.
preds = fill(-1, n)
for i in 1:n
    for t in T_L
        if value(z[i,t]) > 0.5
            k_idx = findfirst(k -> value(c[k,t]) > 0.5, 1:K)
            preds[i] = k_idx !== nothing ? classes[k_idx] : -1
            break
        end
    end
end

# Total error from the L_err variables (sum over all leaves)
total_err = round(Int, sum(value.(L_err)))
acc = 100.0 * (1 - total_err / n)

# Confusion matrix entries (positive class = Malignant = 0):
# TP: malignant correctly predicted malignant
# TN: benign correctly predicted benign
# FP: benign wrongly predicted malignant  (false alarm)
# FN: malignant wrongly predicted benign  (missed cancer — most dangerous)
TP  = sum((preds .== 0) .& (y_raw .== 0))
TN  = sum((preds .== 1) .& (y_raw .== 1))
FP  = sum((preds .== 0) .& (y_raw .== 1))
FN  = sum((preds .== 1) .& (y_raw .== 0))

push!(lines, "")
push!(lines, "─"^60)
push!(lines, "  PERFORMANCE  (Positive = Malignant = 0)")
push!(lines, "─"^60)
push!(lines, @sprintf("  Accuracy    : %.2f %%  (%d/%d correct)", acc, n-total_err, n))
# Sensitivity = TP/(TP+FN): proportion of actual malignant cases caught
push!(lines, @sprintf("  Sensitivity : %.2f %%  (malignant caught)", TP+FN>0 ? 100.0*TP/(TP+FN) : 0.0))
# Specificity = TN/(TN+FP): proportion of actual benign cases correctly cleared
push!(lines, @sprintf("  Specificity : %.2f %%  (benign cleared)",   TN+FP>0 ? 100.0*TN/(TN+FP) : 0.0))
push!(lines, "")
push!(lines, "  Confusion Matrix:")
push!(lines, "                Pred Mal(0)   Pred Ben(1)")
push!(lines, @sprintf("  Actual Mal(0)   %5d        %5d", TP, FN))
push!(lines, @sprintf("  Actual Ben(1)   %5d        %5d", FP, TN))

# ── Dataset & feature info — least critical, placed last ─────
push!(lines, "")
push!(lines, "─"^60)
push!(lines, "  DATASET & FEATURES")
push!(lines, "─"^60)
push!(lines, "  Full dataset : $N_full samples  →  stratified subsample: $n")
push!(lines, "  Classes      : Malignant(0)=$(sum(y_raw.==0)),  Benign(1)=$(sum(y_raw.==1))")
push!(lines, "  Features (p=$p, top Fisher score):")
for (r, fi) in enumerate(feat_idx)
    push!(lines, @sprintf("    %d. %-26s  %.4f", r, all_feature_names[fi], scores[fi]))
end
push!(lines, "="^60)

# Single println call — Jupyter cannot partially truncate one call
println(join(lines, "\n"))

  OCT — Breast Cancer  |  D=2  p=5  n=150
  Solver Status : OPTIMAL
  Build Time    : 2.22 s  (constraint generation)
  Solve Time    : 112.10 s  (1.87 min)
  Total Time    : 117.06 s  (1.95 min)  <- matches Jupyter runtime
  Objective Val : 0.026967
  MIP Gap       : 0.865 %

────────────────────────────────────────────────────────────
  TREE STRUCTURE
────────────────────────────────────────────────────────────
  Node 1 │ concave_pts_mean         ≤ 0.0555
  Node 2 │ area_worst               ≤ 876.5000
  Node 3 │ concave_pts_worst        ≤ 0.1374
  Leaf 4 │ → Benign     (n=88)
  Leaf 5 │ → Malignant  (n=10)
  Leaf 6 │ → Benign     (n=6)
  Leaf 7 │ → Malignant  (n=46)

────────────────────────────────────────────────────────────
  PERFORMANCE  (Positive = Malignant = 0)
────────────────────────────────────────────────────────────
  Accuracy    : 97.33 %  (146/150 correct)
  Sensitivity : 96.43 %  (malignant caught)
  Specificity : 97.87 %  (benign cleared)

  Confusion Matrix:
        